In [9]:
#!/usr/bin/env python3
import os
import sys
import numpy as np
import pandas as pd
from skbio.stats.distance import permanova, permdisp
from skbio.diversity import beta_diversity
from statsmodels.stats.multitest import multipletests

# -------------------- Parameters & I/O --------------------

test_mode = True
work_name = "Full_wo55"
work_name = "Vaginal"
if test_mode:
    params_permutations     = 200
    params_pvalue_threshold  = 0.25
    params_ignore_cols = ['Barcode','Fecha_de_la_librería', 'Fecha_de_extracción_de_ADN', 'Número_Madre', 'estrés_T2_logistico.c32', 'per.Número_de_reads', 'per.Estrés_percibido_T1', 'per.Estrés_percibido_T2'] + ['IMC.c25',
        'per.Adiponectina_M2',
        'per.Cortisol_T2',
        'per.Frac.peso-talla.12m',
        'per.Ganancia_peso_materno',
        'per.IMC_materna',
        'per.IgA_M2',
        'per.IgA_MF_M2',
        'per.IgM_M2']


if work_name == "Full_wo55":
    input_taxa_ab_df_path   = "/mnt/Disk04/jpereira/Results/MReport/Cecilia_HS_1/Data/ARN16S/Classify_16S/taxa-sequences.csv"
    input_metadata_df_path  = '/mnt/Disk04/jpereira/MReport-test/Metadata/metadata_ceciliaHS_full_30-09-24.tsv'
    output_permanova_csv    = "/mnt/Disk04/jpereira/Results/MReport/Cecilia_HS_1/Data/ARN16S/PERMANOVA/permanova_results.csv"
    output_permanova_html   = "/mnt/Disk04/jpereira/Results/MReport/Cecilia_HS_1/Data/ARN16S/PERMANOVA/permanova_results.html"
elif work_name == "Cesarea":
    input_taxa_ab_df_path   = "/mnt/Disk04/jpereira/Results/MReport/Cecilia_HS_C/Data/ARN16S/Classify_16S/taxa-sequences.csv"
    input_metadata_df_path  = '/mnt/Disk04/jpereira/MReport-test/Metadata/metadata_ceciliaHS_C_30-09-24.tsv'
    output_permanova_csv    = "/mnt/Disk04/jpereira/Results/MReport/Cecilia_HS_C/Data/ARN16S/PERMANOVA/permanova_results.csv"
    output_permanova_html   = "/mnt/Disk04/jpereira/Results/MReport/Cecilia_HS_C/Data/ARN16S/PERMANOVA/permanova_results.html"
elif work_name == "Vaginal":
    input_taxa_ab_df_path   = "/mnt/Disk04/jpereira/Results/MReport/Cecilia_HS_V/Data/ARN16S/Classify_16S/taxa-sequences.csv"
    input_metadata_df_path  = '/mnt/Disk04/jpereira/MReport-test/Metadata/metadata_ceciliaHS_V_30-09-24.tsv'
    output_permanova_csv    = "/mnt/Disk04/jpereira/Results/MReport/Cecilia_HS_V/Data/ARN16S/PERMANOVA/permanova_results.csv"
    output_permanova_html   = "/mnt/Disk04/jpereira/Results/MReport/Cecilia_HS_V/Data/ARN16S/PERMANOVA/permanova_results.html"


else:
    input_taxa_ab_df_path  = sys.argv[1]
    input_metadata_df_path = sys.argv[2]
    output_permanova_csv   = sys.argv[3]

# Create output directory if needed
os.makedirs(os.path.dirname(output_permanova_csv), exist_ok=True)


In [3]:
def color_code(value, thresholds = [0.01, 0.05, 0.10], colors = ['🟦','🟩','🟨','🟥']):
    """
    Assigns a color based on a list of thresholds.

    Parameters:
        value (float): The numeric value to evaluate (e.g., p-value).
        thresholds (list): A list of thresholds in increasing order.
        colors (list): A list of N+1 colors, where N is the number of thresholds.

    Returns:
        str: The corresponding color emoji.
    """
    for i, threshold in enumerate(thresholds):
        if value <= threshold:
            return colors[i]
    return colors[-1]


In [10]:

# -------------------- Load Data --------------------

taxa_df     = pd.read_csv(input_taxa_ab_df_path, sep=",", header=0)
meta_df     = pd.read_csv(input_metadata_df_path, sep="\t", header=0)

# Filter out low‐confidence assignments if you wish:
taxa_df = taxa_df[taxa_df['confidence'] > 0.9]
taxa_df = taxa_df.rename(columns={'gender' : 'genus'})
taxa_df = taxa_df.rename(columns={'specie' : 'species'})

# -------------------- Define Taxonomic Levels --------------------

taxonomic_levels = ['phylo', 'class', 'order', 'family', 'genus', 'species']

# Identify categorical columns in metadata
cat_cols = meta_df.select_dtypes(include=['object', 'category']).columns.tolist()

# Identify categorical columns in metadata
match_regex = 'categorical|#q2:types'
meta_cat_df = meta_df.loc[:, meta_df.iloc[0].str.match(match_regex) ]
meta_cat_df = meta_cat_df.drop(0) #Droping column subtitle

meta_cat_df = meta_cat_df.set_index("sample-id")
meta_df = meta_df.set_index("sample-id")

sample_ids = list(meta_cat_df.index)
cat_cols = meta_cat_df.columns
# -------------------- PERMANOVA Loop --------------------

results = []
disp_results = []
for level in taxonomic_levels:
    # 1) Subset and clean up the taxonomy column
    tmp = taxa_df[sample_ids + [level]].copy()
    tmp[level] = tmp[level].replace('--', np.nan)
    tmp = tmp.dropna(subset=[level])

    # 2) Sum counts per taxon
    counts = tmp.groupby(level)[sample_ids].sum()

    # 3) Compute relative abundances per sample
    relab = counts.div(counts.sum(axis=0), axis=1).T  # shape = (samples × taxa)

    # 4) Build Bray–Curtis distance matrix
    dm = beta_diversity(metric="braycurtis",
                        counts=relab.values,
                        ids=relab.index)

    # 5) Run PERMANOVA for each categorical variable
    for col in cat_cols:
        if col in params_ignore_cols:
            continue

        grouping = meta_df.loc[relab.index, col]

        # Skip if fewer than 2 groups
        if grouping.nunique() < 2:
            continue

        # Skip if any group has fewer than 2 samples
        group_sizes = grouping.value_counts()
        if (group_sizes < 2).any():
            continue

        res = permanova(distance_matrix=dm,
                        grouping=grouping,
                        permutations=params_permutations)
        disp_res = permdisp(distance_matrix=dm,
                 grouping=grouping,
                 permutations=params_permutations)
        
        has_valid_f = not np.isnan(res['test statistic'])
    
        if has_valid_f:
            results.append({
                'Taxonomic_Level': level,
                'Metadata_Column': col,
                'Pseudo-F': res['test statistic'],
                'p-value.Pseudo-F': res['p-value'],
                'F-disp' : disp_res['test statistic'],
                'p-value.F-disp': disp_res['p-value'],
                'N_Groups': grouping.nunique(),
                'Permutations': res['number of permutations'],
                })

# -------------------- Save Results --------------------

res_df = pd.DataFrame(results)

# Adjust PERMANOVA p-values
adj_pvals = multipletests(res_df['p-value.Pseudo-F'], method='fdr_bh')[1]
res_df.insert(3,'p-adj.Pseudo-F', adj_pvals)

permanova_col = res_df['p-adj.Pseudo-F'].apply( lambda x: color_code(x))
permadisp_col = res_df['p-value.F-disp'].apply( lambda x: color_code(x, thresholds=[0.05, 0.10], colors=['🟥', '🟨', '🟩']))

# Insert them at the beginning
res_df.insert(0, 'PERMADISP', permadisp_col)
res_df.insert(0, 'PERMANOVA', permanova_col)

final_res_df = res_df[res_df['p-adj.Pseudo-F'] < params_pvalue_threshold]

final_res_df = final_res_df.sort_values(['Metadata_Column','Taxonomic_Level']).set_index(['Metadata_Column','Taxonomic_Level'])
res_df.to_csv(output_permanova_csv, index=True)

# Create directory to save the html file
table_html_dir = os.path.dirname(output_permanova_html)
os.makedirs(table_html_dir, exist_ok=True)

# - - - - - - Save Results in a HTML file - - - - - - - -

# Making a html file with the DataFrame
html = final_res_df.to_html(escape=False, classes='table table-striped table-hover')

# Saving html file
with open(output_permanova_html, 'w') as file:
    file.write("""
<style>
  .table {
    border-collapse: collapse;
    width: 100%;
    font-family: 'Arial', sans-serif; /* or the font of your choice */
  }
  .table th,
  .table td {
    border: 1px solid #ddd; /* Lighter border color as seen in the image */
    padding: 10px; /* More padding for a spacious look */
    text-align: left;
  }
  .table thead th {
    background-color: #f8f8f8; /* Light grey background for the header */
    color: #333; /* Dark text for contrast */
    font-weight: bold;
  }
  .table-striped tbody tr:nth-of-type(odd) {
    background-color: #f9f9f9; /* Zebra striping color, adjust as needed */
  }
  .table-hover tbody tr:hover {
    background-color: #f1f1f1; /* Hover color, adjust as needed */
  }
</style>
""")
    file.write(html)

print('Saved PERMANOVA results in: ')
print(f'{output_permanova_csv}')
print(f'{output_permanova_html}')

#print(f"PERMANOVA results saved to {output_permanova_csv}")


/mnt/Disk04/jpereira/miniconda3/envs/MReport-test2/lib/python3.9/site-packages/skbio/stats/ordination/_principal_coordinate_analysis.py:146: RuntimeWarning: The result contains negative eigenvalues. Please compare their magnitude with the magnitude of some of the largest positive eigenvalues. If the negative ones are smaller, it's probably safe to ignore them, but if they are large in magnitude, the results won't be useful. See the Notes section for more details. The smallest eigenvalue is -0.07660506775292968 and the largest is 1.3732324182219533.
  warn(
/mnt/Disk04/jpereira/miniconda3/envs/MReport-test2/lib/python3.9/site-packages/skbio/stats/ordination/_principal_coordinate_analysis.py:146: RuntimeWarning: The result contains negative eigenvalues. Please compare their magnitude with the magnitude of some of the largest positive eigenvalues. If the negative ones are smaller, it's probably safe to ignore them, but if they are large in magnitude, the results won't be useful. See the N

Saved PERMANOVA results in: 
/mnt/Disk04/jpereira/Results/MReport/Cecilia_HS_V/Data/ARN16S/PERMANOVA/permanova_results.csv
/mnt/Disk04/jpereira/Results/MReport/Cecilia_HS_V/Data/ARN16S/PERMANOVA/permanova_results.html


/mnt/Disk04/jpereira/miniconda3/envs/MReport-test2/lib/python3.9/site-packages/scipy/stats/_axis_nan_policy.py:563: ConstantInputWarning: Each of the input arrays is constant; the F statistic is not defined or infinite
  res = hypotest_fun_out(*samples, axis=axis, **kwds)
/mnt/Disk04/jpereira/miniconda3/envs/MReport-test2/lib/python3.9/site-packages/scipy/stats/_axis_nan_policy.py:563: ConstantInputWarning: Each of the input arrays is constant; the F statistic is not defined or infinite
  res = hypotest_fun_out(*samples, axis=axis, **kwds)
/mnt/Disk04/jpereira/miniconda3/envs/MReport-test2/lib/python3.9/site-packages/scipy/stats/_axis_nan_policy.py:563: ConstantInputWarning: Each of the input arrays is constant; the F statistic is not defined or infinite
  res = hypotest_fun_out(*samples, axis=axis, **kwds)
/mnt/Disk04/jpereira/miniconda3/envs/MReport-test2/lib/python3.9/site-packages/scipy/stats/_axis_nan_policy.py:563: ConstantInputWarning: Each of the input arrays is constant; the F